# **Topic: Support Ticket Intelligence system**

Where aim is to deliver an intelligent system that can reduce the workload to provide an efficient solution with ticket monitoring system

The overall stucture summarised as:-

Ticket
   │
   ▼
Topic Classification
   │
   ▼
Risk Detection
   │
   ▼
Similarity Search
   │
   ▼
Ticket Summary
   │
   ▼
Suggested Solution
   │
   ▼
Team Routing
   │
   ▼
Final Report

# Import Libraries

In [ ]:
!pip install evaluate
!pip install groq
!pip install rouge-score

In [ ]:
# for EDA
import pandas as pd
import numpy as np
from datasets import load_dataset
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# for Ticket classfiication
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import torch
import evaluate
from groq import Groq # to import LLaMA LLM free

# For Similarity search
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from rouge_score import rouge_scorer

# for Risk detection
import json
import re




import warnings
warnings.filterwarnings("ignore")

## Load Dataset

In [ ]:
dataset = load_dataset(
    "Console-AI/IT-helpdesk-synthetic-tickets"
)

# EDA the dataset

Convert to Dataframe

In [ ]:
df = dataset["train"].to_pandas()

Manifest basic dataset information

In [ ]:
print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

Checking for Missing Values

In [ ]:
df.isnull().sum()

Checking for datatypes

In [ ]:
df.info()

Create Main ticket text column

In [ ]:
df["ticket_text"] = (
    df["subject"].fillna("")
    + " "
    + df["description"].fillna("")
)

Quick dataset statistics

In [ ]:
df.describe(include="object").T

Ticket Length

In [ ]:
df["ticket_length"] = (
    df["ticket_text"]
    .str.split()
    .apply(len)
)

Showing basic statistics

In [ ]:
df["ticket_length"].describe()

Histogram

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    df["ticket_length"],
    bins=30
)

plt.title("Ticket Length Distribution")
plt.xlabel("Words")
plt.ylabel("Count")

plt.show()

Word Cloud

In [ ]:
all_text = " ".join(df["ticket_text"])

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white"
).generate(all_text)

plt.figure(figsize=(15,7))
plt.imshow(wordcloud)
plt.axis("off")
plt.show()

before coding any classifier, its better to inspect by understanding labels

In [ ]:
for col in df.columns:
    print(col)

To know from the df:
*   Category column name
*   Priority column name
*   Available labels

In [ ]:
for col in df.columns:
    print("\n")
    print("="*50)
    print(col)
    print(df[col].value_counts().head())

Inspect the Class distribution and visualise it

In [ ]:
print("Category Distribution\n")
print(df["category"].value_counts())

print("\n")

print("Priority Distribution\n")
print(df["priority"].value_counts())

plt.figure(figsize=(10,5))

sns.countplot(
    data=df,
    x="category",
    order=df["category"].value_counts().index
)

plt.xticks(rotation=45)
plt.title("Category Distribution")

plt.show()

# Ticket Topic Classification
Goal is to compare Tf-IDF + Logistic Regression, with DistilBERT and Zero-Shot LLM

## Using TF-IDF + Logistic Regression

Remove Rare classes

In [ ]:
category_counts = df["category"].value_counts()

valid_categories = category_counts[
    category_counts >= 2
].index

df = df[
    df["category"].isin(valid_categories)
].copy()

Encode labels

In [ ]:
label_encoder = LabelEncoder()

df["category_encoded"] = label_encoder.fit_transform(
    df["category"]
)

print(label_encoder.classes_)

train test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["ticket_text"],
    df["category_encoded"],
    test_size=0.2,
    random_state=42,
    stratify=df["category_encoded"]
)

Train using baseline model

TF+IDF + Logistic Regression


In [ ]:
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            max_features=5000
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=1000
        )
    )
])

In [ ]:
baseline_model.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = baseline_model.predict(X_test)

Evaluate

In [ ]:
baseline_accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", baseline_accuracy)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)
plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("TF-IDF Classifier")

plt.show()

Real-life prediction function

In [ ]:
def classify_ticket(text):

    pred = baseline_model.predict([text])[0]

    category = label_encoder.inverse_transform(
        [pred]
    )[0]

    return category

In [ ]:
sample_ticket = """
VPN disconnects every few minutes and
employees cannot access internal servers.
"""
classify_ticket(sample_ticket)

testing with more examples

In [ ]:
examples = [
    "Cannot login to my account",
    "Laptop screen is flickering",
    "Outlook keeps crashing",
    "VPN not connecting",
    "Suspicious login detected"
]

for ex in examples:

    print("Ticket:", ex)

    print(
        "Category:",
        classify_ticket(ex)
    )

    print("-"*50)

## with DistilBERT fine tuning

create label mapping

In [ ]:
num_labels = len(label_encoder.classes_)

id2label = {
    i: label
    for i, label in enumerate(label_encoder.classes_)
}

label2id = {
    label: i
    for i, label in enumerate(label_encoder.classes_)
}

print(id2label)

create train/test dataframes

In [ ]:
train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

Convert to hugging face dataset

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

load distilbert based tokenizer

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = DistilBertTokenizerFast.from_pretrained(
    model_name
)

tokenization function

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

apply tokenization

In [ ]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Remove unnecessary columns

In [ ]:
train_dataset = train_dataset.remove_columns(
    ["text"]
)

test_dataset = test_dataset.remove_columns(
    ["text"]
)

Convert labels

In [ ]:
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

test_dataset = test_dataset.rename_column(
    "label",
    "labels"
)

load distilBERT model

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Evaluation Matrics

In [ ]:
accuracy_metric = evaluate.load(
    "accuracy"
)

f1_metric = evaluate.load(
    "f1"
)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

Training Arguments

In [ ]:
training_args = TrainingArguments(

    output_dir="./distilbert_ticket_classifier",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=5,

    weight_decay=0.01,

    load_best_model_at_end=True,

    logging_steps=10,

    report_to="none"
)

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)
trainer.train()

In [ ]:
results = trainer.evaluate()

results

In [ ]:
distilbert_accuracy = results['eval_accuracy']

Detailed CLassification Report

In [ ]:
predictions = trainer.predict(
    test_dataset
)
y_pred_bert = np.argmax(
    predictions.predictions,
    axis=1
)
print(
    classification_report(
        y_test,
        y_pred_bert,
        target_names=label_encoder.classes_,
        labels=range(num_labels) # Explicitly provide all possible labels
    )
)

Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_bert
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("DistilBERT Classifier confusion matrix")

plt.show()

examplified on test dataset

In [ ]:
def classify_ticket_bert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():

        outputs = model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    return id2label[prediction]

classify_ticket_bert(
    "VPN keeps disconnecting and users cannot reach internal servers"
)

## Next using Zero-shot LLM Classification
that aim to ask LLM to predict ticket catagory instead of training a model

In [ ]:
#using actual dataset labels
categories = sorted(
    df["category"].unique()
)

print(categories)

setup Grouq Client

In [129]:
client = Groq(api_key="use_your_api")

Create a classification prompt

In [ ]:
def classify_ticket_llm(ticket_text):

    prompt = f"""
You are an IT Helpdesk Analyst.

Classify the following ticket into ONE category.

Allowed Categories:
{categories}

Ticket:
{ticket_text}

Return ONLY the category name.
"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content.strip()

test

In [ ]:
sample_ticket = """
VPN disconnects every few minutes and
employees cannot access internal systems.
"""

classify_ticket_llm(sample_ticket)

test multiple examples

In [ ]:
examples = [

    "Cannot login to my company account",

    "Laptop display is flickering",

    "Outlook keeps crashing",

    "VPN not connecting",

    "Suspicious login detected"
]

In [ ]:
for ticket in examples:

    prediction = classify_ticket_llm(ticket)

    print(ticket)

    print("Prediction:", prediction)

    print("-"*50)

In [ ]:
sample_df = df.sample(
    50,
    random_state=42
)

In [ ]:
llm_predictions = []

for ticket in sample_df["ticket_text"]:
    llm_predictions.append(classify_ticket_llm(ticket))

accuracy

In [ ]:
llm_accuracy = accuracy_score(
    sample_df["category"],
    llm_predictions
)
print("Accuracy:", llm_accuracy)

Classification report



In [ ]:
print(
    classification_report(
        sample_df["category"],
        llm_predictions
    )
)

Do a comparison between each outputs to show the final result for ticket classification

In [ ]:
comparison_df = pd.DataFrame({

    "Model":[
        "TF-IDF + Logistic Regression",
        "DistilBERT",
        "Zero-Shot LLM"
    ],

    "Accuracy":[
        baseline_accuracy,
        distilbert_accuracy,
        llm_accuracy
    ]
})

print(comparison_df)

In [ ]:
comparison_df.sort_values(
    "Accuracy",
    ascending=False
)
plt.figure(figsize=(8,4))

sns.barplot(
    data=comparison_df,
    x="Model",
    y="Accuracy"
)

plt.title("Category Classification Performance")

plt.xticks(rotation=20)

plt.show()

In [ ]:
def compare_all_models(ticket):

    print("Ticket:")
    print(ticket)

    print("\nTF-IDF:")
    print(classify_ticket(ticket))

    print("\nDistilBERT:")
    print(classify_ticket_bert(ticket))

    print("\nLLM:")
    print(classify_ticket_llm(ticket))

compare_all_models("VPN not connecting")

# Ticket Similarity Search
The purpose is to introduce embedding, semantic search, vector representations, retrival systems, foundation of RAG

**Loading the embedding model**

all-MiniLM-L6-v2 is considered because fast, lightweight, excellent semantic search performance, popular in production

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Generate ticket Embeddings

In [ ]:
ticket_texts = df["ticket_text"].tolist()
ticket_embeddings = embedding_model.encode(
    ticket_texts,
    show_progress_bar=True
)

checking shape

In [ ]:
ticket_embeddings.shape

Create Similarity function

In [ ]:
def find_similar_tickets(

    query,

    top_k=5
):

    query_embedding = embedding_model.encode(
        [query]
    )

    similarities = cosine_similarity(
        query_embedding,
        ticket_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:

        results.append({

            "similarity": round(
                similarities[idx],
                3
            ),

            "category":
            df.iloc[idx]["category"],

            "priority":
            df.iloc[idx]["priority"],

            "ticket":
            df.iloc[idx]["ticket_text"][:250]
        })

    return pd.DataFrame(results)

test with samples

In [ ]:
query = """
VPN disconnects every few minutes and
employees cannot access company resources.
"""
find_similar_tickets(
    query
)

To look how semantic similarity behaves

In [ ]:
sample_query = """
VPN disconnecting repeatedly
"""
query_embedding = embedding_model.encode(
    [sample_query]
)
scores = cosine_similarity(
    query_embedding,
    ticket_embeddings
)[0]
plt.figure(figsize=(8,4))

sns.histplot(
    scores,
    bins=30
)

plt.title(
    "Similarity Score Distribution"
)

plt.show()

Retrieve full ticket details

function will be useful for risk detectiona nd solution generation, because those modules can look at similar historical tickets

In [ ]:
def retrieve_similar_tickets(

    query,

    top_k=5
):

    query_embedding = embedding_model.encode(
        [query]
    )

    similarities = cosine_similarity(
        query_embedding,
        ticket_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    return df.iloc[top_indices][
        [
            "subject",
            "category",
            "priority",
            "description"
        ]
    ]

Evaluate retrieval quality

In [ ]:
correct = 0

total = 50

In [ ]:
sample_indices = np.random.choice(
    len(df),
    total,
    replace=False
)

In [ ]:
for idx in sample_indices:

    query = df.iloc[idx]["ticket_text"]

    query_embedding = embedding_model.encode(
        [query]
    )

    similarities = cosine_similarity(
        query_embedding,
        ticket_embeddings
    )[0]

    similarities[idx] = -1

    nearest = similarities.argmax()

    if (
        df.iloc[idx]["category"]
        ==
        df.iloc[nearest]["category"]
    ):
        correct += 1

In [ ]:
retrieval_accuracy = correct / total

print(
    f"Category Match Accuracy: {retrieval_accuracy:.2%}"
)

save the embeddings to prevent recomputations

**In this section, Cosine_similarity() is used due to small dataset, its suggested to use FAISS Vector Database for more industry-oriented response**


import faiss

@faiss requires float32

ticket_embeddings = np.array(
    ticket_embeddings
).astype("float32") @ covert embeddings

dimension = ticket_embeddings.shape[1] @ create index

index.add(ticket_embeddings) @add vector

def faiss_search(query,top_k=5):
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")
    distances, indices = index.search(query_embedding,top_k)

    return df.iloc[
        indices[0]
    ][
        [
            "subject",
            "category",
            "priority"
        ]
    ]

This is the same retrieval pattern used in:
*   OpenAI retrieval systems
*   LangChain applications
*   Enterprise RAG chatbots
*   Internal knowledge assistants

# Ticket Summarisation
The purpose is to demonstrate prompt engineering, LLM integration, text summarization and Rouge (evaluation metrics)

Create summary functions

In [ ]:
def summarise_ticket(ticket_text):

    prompt = f"""
You are an IT Helpdesk Analyst.

Summarise the following ticket in ONE concise sentence.

Ticket:
{ticket_text}

Return only the summary.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content.strip()

Test on One ticket

In [ ]:
sample_ticket = df.iloc[0]["ticket_text"]

print(sample_ticket)

In [ ]:
summary = summarise_ticket(sample_ticket)

print(summary)

Test on Multiple tickets

In [ ]:
for i in range(5):

    ticket = df.iloc[i]["ticket_text"]

    summary = summarise_ticket(ticket)

    print("="*80)

    print("ORIGINAL\n")
    print(ticket[:500])

    print("\nSUMMARY\n")
    print(summary)

Create summary column and amend into it

In [ ]:
sample_df = df.sample(
    25,
    random_state=42
).copy()
sample_df["summary"] = sample_df[
    "ticket_text"
].apply(
    summarise_ticket
)

Review results

In [ ]:
sample_df[
    [
        "ticket_text",
        "summary"
    ]
].head()

Compression ratio

In [ ]:
sample_df["original_words"] = (
    sample_df["ticket_text"]
    .str.split()
    .str.len()
)
sample_df["summary_words"] = (
    sample_df["summary"]
    .str.split()
    .str.len()
)
sample_df["compression_ratio"] = (
    sample_df["summary_words"]
    /
    sample_df["original_words"]
)
sample_df[
    "compression_ratio"
].describe()

visualise

In [ ]:
plt.figure(figsize=(8,4))

sns.histplot(
    sample_df["compression_ratio"],
    bins=15
)

plt.title(
    "Summary Compression Ratio"
)

plt.show()

Temperature Experiment


deterministic summary

In [ ]:
def summarise_temp(
    ticket_text,
    temperature
):

    prompt = f"""
Summarise this IT ticket in one sentence.

Ticket:
{ticket_text}
"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],

        temperature=temperature
    )

    return response.choices[0].message.content.strip()
ticket = df.iloc[10]["ticket_text"]

In [ ]:
for temp in [0,0.3,0.7,1.0]:

    print("="*80)

    print("Temperature:", temp)

    print(
        summarise_temp(
            ticket,
            temp
        )
    )

ROUGE evaluation

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1","rougeL"],
    use_stemmer=True
)

In [ ]:
reference = df.iloc[0]["subject"]

generated = summarise_ticket(
    df.iloc[0]["ticket_text"]
)

In [ ]:
scores = scorer.score(
    reference,
    generated
)

scores

Evaluate on 25 tickets

In [ ]:
rouge1_scores = []
rougeL_scores = []

In [ ]:
for idx in sample_df.index:

    reference = df.loc[
        idx,
        "subject"
    ]

    generated = sample_df.loc[
        idx,
        "summary"
    ]

    scores = scorer.score(
        reference,
        generated
    )

    rouge1_scores.append(
        scores["rouge1"].fmeasure
    )

    rougeL_scores.append(
        scores["rougeL"].fmeasure
    )

In [ ]:
print(
    "Average ROUGE-1:",
    np.mean(rouge1_scores)
)

print(
    "Average ROUGE-L:",
    np.mean(rougeL_scores)
)

Summarise quality examples

In [ ]:
results_df = sample_df[
    [
        "subject",
        "summary"
    ]
].head(10)
print(results_df)

# Risk detection
to detemine the urgency of the problem

Classify the levels of risk

In [ ]:
risk_levels = ["Low","Medium","High","Critical"]

create Risk detection function

In [ ]:
def detect_risk(ticket_text):

    prompt = f"""
You are a senior IT incident analyst.

Analyse the ticket and determine the operational risk.

Risk Levels:
- Low
- Medium
- High
- Critical

Rules:

Low:
- Single user issue
- Minor inconvenience
- No business impact

Medium:
- Department level issue
- Productivity affected

High:
- Multiple users affected
- Core application impacted

Critical:
- Security breach
- Company-wide outage
- Data loss risk

Return JSON only:

{{
    "risk_level":"",
    "reason":"",
    "recommended_action":""
}}

Ticket:
{ticket_text}
"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],

        temperature=0
    )

    content = response.choices[0].message.content.strip()

    # Remove markdown if present
    content = content.replace("```json", "")
    content = content.replace("```", "")
    content = content.strip()

    match = re.search(
        r"\{.*\}",
        content,
        re.DOTALL
    )

    if match:
        return json.loads(match.group())

    return {
        "risk_level": "Unknown",
        "reason": "Parsing Failed",
        "recommended_action": "Manual Review"
    }

applying the sample tickets

In [ ]:
risk_sample = df.sample(
    30,
    random_state=42
).copy()
risk_result=[]
for ticket in risk_sample["ticket_text"]:
  result = detect_risk(ticket)
  risk_result.append(result)

Convert to dataframe

In [ ]:
risk_df = pd.DataFrame(risk_result)
risk_df.head()

merge back

In [ ]:
risk_sample = pd.concat(
    [
        risk_sample.reset_index(drop=True),
        risk_df
    ],
    axis=1
)

Risk DIstribution

In [ ]:
risk_sample["risk_level"].value_counts()

#bar chart
plt.figure(figsize=(8,4))

sns.countplot(
    data=risk_sample,
    x="risk_level",
    order=[
        "Low",
        "Medium",
        "High",
        "Critical"
    ]
)

plt.title(
    "Risk Level Distribution"
)

plt.show()

#pie chart
risk_sample[
    "risk_level"
].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%"
)

plt.ylabel("")

plt.title(
    "Risk Distribution"
)

plt.show()

risk examples table

In [ ]:
risk_sample[["subject","risk_level","reason"]].head(10)

Create reusable function
to integrate this into full pipeline

In [ ]:
def risk_score(ticket_text):

    result = detect_risk(
        ticket_text
    )

    return result["risk_level"]

**Optional Rule-Based baseline**

just like we compared classifiers, we can compare risk detections

In [ ]:
def rule_based_risk(ticket):

    ticket = ticket.lower()

    if any(
        word in ticket
        for word in [
            "outage",
            "breach",
            "attack",
            "ransomware"
        ]
    ):
        return "Critical"

    elif any(
        word in ticket
        for word in [
            "multiple users",
            "department",
            "vpn"
        ]
    ):
        return "High"

    elif any(
        word in ticket
        for word in [
            "access",
            "setup"
        ]
    ):
        return "Low"

    return "Medium"

#test
ticket = """Several employees cannot login to email"""
rule_based_risk(ticket)

# Solution Recommendation
RAG-Lite

to justify "how do i fix the problem?"

by using approch

New Ticket
     ↓
Similarity Search
     ↓
Top Similar Tickets
     ↓
LLM
     ↓
Suggested Solution

Improve the similarity retrieval

In [ ]:
def retrieve_ticket_context(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query]
    )

    similarities = cosine_similarity(
        query_embedding,
        ticket_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    contexts = []

    for idx in top_indices:

        ticket = df.iloc[idx]

        contexts.append({
            "subject": ticket["subject"],
            "description": ticket["description"],
            "category": ticket["category"],
            "priority": ticket["priority"]
        })

    return contexts

Create context formatter

In [ ]:
def format_context(contexts):

    formatted = ""

    for i, ticket in enumerate(contexts, start=1):

        formatted += f"""
Historical Ticket {i}

Category: {ticket['category']}
Priority: {ticket['priority']}
Subject: {ticket['subject']}
Description: {ticket['description']}

-----------------------
"""
    return formatted

Testing

In [ ]:
query = """
VPN disconnects every few minutes and
employees cannot access internal resources.
"""

contexts = retrieve_ticket_context(query)

print(
    format_context(contexts)
)

Solution Generation function

Complete LLM call

In [ ]:
def parse_llm_json(response_text):
    """
    Safely parse LLM JSON responses.
    """

    try:
        return json.loads(response_text)

    except Exception:

        # Remove markdown code fences
        cleaned = re.sub(
            r"```json|```",
            "",
            response_text
        ).strip()

        try:
            return json.loads(cleaned)

        except Exception:

            # Extract JSON object if surrounded by text
            match = re.search(
                r"\{.*\}",
                cleaned,
                re.DOTALL
            )

            if match:

                try:
                    return json.loads(
                        match.group()
                    )

                except Exception:
                    pass

            return {
                "issue_summary": "Parsing Failed",
                "recommended_solution": response_text,
                "confidence": "Unknown"
            }

In [ ]:
def generate_solution(ticket_text):

    contexts = retrieve_ticket_context(
        ticket_text,
        top_k=3
    )

    context_text = format_context(
        contexts
    )

    prompt = f"""
You are an experienced IT Helpdesk Engineer.

A new support ticket has been submitted.

Use the historical tickets below as context.

Historical Tickets:
{context_text}

New Ticket:
{ticket_text}

Analyze the issue and recommend a solution.

IMPORTANT:
- Return ONLY valid JSON
- Do not use markdown
- Do not add explanations
- Do not wrap the response in ```json

Return exactly this format:

{{
    "issue_summary":"",
    "recommended_solution":"",
    "confidence":"Low|Medium|High"
}}
"""

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    response_text = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    return parse_llm_json(
        response_text
    )

test

In [ ]:
ticket = """
VPN disconnects every few minutes and
employees cannot access company resources.
"""
result= generate_solution(ticket)
print (result)

test with multiple examples

In [ ]:
examples = [

    "Cannot login to Outlook account",

    "Laptop screen flickering",

    "VPN disconnects repeatedly",

    "Google Calendar not syncing",

    "Suspicious login activity detected"
]
for ticket in examples:

    print("="*80)

    print("TICKET:")
    print(ticket)

    print()

    result = generate_solution(ticket)

    print(result)

create solution dataset, by analysing 20 tickets

In [ ]:
solution_sample = df.sample(
    20,
    random_state=42
).copy()
solution_result=[]
for ticket in solution_sample["ticket_text"]:
  result = generate_solution(ticket)
  solution_result.append(result)

convert to dataframe and show

In [ ]:
solution_df = pd.DataFrame(solution_result)
solution_df.head()

Merge Results

In [ ]:
solution_sample = pd.concat(
    [
        solution_sample.reset_index(drop=True),
        solution_df
    ],
    axis=1
)

Confidence distribution

In [ ]:
solution_sample["confidence"].value_counts()
sns.countplot(data=solution_sample,x="confidence")

plt.title("Confidence Distribution")
plt.show()

Showcase Examples

In [ ]:
solution_sample[
    [
        "subject",
        "issue_summary",
        "recommended_solution",
        "confidence"
    ]
].head(10)

# Team Routing Recommendation



Simple rule-based routing

In [ ]:
team_mapping = {

    "Network":"Network Operations Team",

    "Software":"Application Support Team",

    "Hardware":"Hardware Support Team",

    "Security":"Cybersecurity Team",

    "Account":"Identity & Access Management Team"
}

Rputing function

In [ ]:
def recommend_team(category):

    return team_mapping.get(
        category,
        "General IT Support Team"
    )

Risk-based Escalation

In [ ]:
def recommend_team_and_priority(
    category,
    risk_level
):

    team = team_mapping.get(
        category,
        "General IT Support Team"
    )

    escalation = "Normal"

    if risk_level == "High":

        escalation = "Priority"

    elif risk_level == "Critical":

        escalation = "Immediate Escalation"

    return {
        "assigned_team": team,
        "escalation": escalation
    }

Example

In [ ]:
recommend_team_and_priority(
    category="Network",
    risk_level="Critical"
)

# Explainable AI
full pipeline of overall operation

In [126]:
def analyse_ticket(ticket_text):

    # Classification
    category = classify_ticket_llm(
        ticket_text
    )

    # Summary
    summary = summarise_ticket(
        ticket_text
    )

    # Risk
    risk_result = detect_risk(
        ticket_text
    )

    # Routing
    routing = recommend_team_and_priority(
        category,
        risk_result["risk_level"]
    )

    # Similar Tickets
    similar_tickets = find_similar_tickets(
        ticket_text,
        top_k=3
    )

    # Solution
    solution = generate_solution(
        ticket_text
    )

    return {

        "category": category,

        "summary": summary,

        "risk_level":
        risk_result["risk_level"],

        "risk_reason":
        risk_result["reason"],

        "assigned_team":
        routing["assigned_team"],

        "escalation":
        routing["escalation"],

        "recommended_solution":
        solution["recommended_solution"],

        "solution_confidence":
        solution["confidence"],

        "similar_tickets":
        similar_tickets.to_dict(
            orient="records"
        )
    }

In [127]:
def display_incident_report(result):

    print("="*70)
    print("           AI HELPDESK INCIDENT REPORT")
    print("="*70)

    print(f"\nCategory           : {result['category']}")
    print(f"Summary            : {result['summary']}")
    print(f"Risk Level         : {result['risk_level']}")
    print(f"Risk Reason        : {result['risk_reason']}")

    print(f"\nAssigned Team      : {result['assigned_team']}")
    print(f"Escalation         : {result['escalation']}")

    print(f"\nRecommended Action :")
    print(result['recommended_solution'])

    print(f"\nConfidence         : {result['solution_confidence']}")

    print("\nTop Similar Tickets")

    for i, ticket in enumerate(
        result["similar_tickets"],
        start=1
    ):

        print(f"\n{i}. Similarity: {ticket['similarity']:.2f}")

        print(f"   Category : {ticket['category']}")

        print(f"   Priority : {ticket['priority']}")

        print(f"   Ticket   : {ticket['ticket'][:120]}")

    print("\n" + "="*70)

Test tie pipeline

In [128]:
ticket = """
VPN disconnects every few minutes and
employees cannot access company resources.
"""

result = analyse_ticket(ticket)

display_incident_report(result)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kv8rz9rse3mrzbh86240tzff` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99921, Requested 165. Please try again in 1m14.304s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}